In [ ]:
def MR_curves(hybrid_sets, dE_values=(0, 25, 50, 100)):

    #Values for Pressure on the Center 
    Pc_vals = np.geomspace(1e-5, 3e3, 260)

    #Plot
    fig2 = plt.figure(figsize=(7,6))

    curves = []          #Store all curves
    headers = []         #headers for txt

    #Pure hadronic curve
    R_had, M_had = [], []     #Void Lists 
    for Pc in Pc_vals:
        Rf, Mf = integrate_star_hadronic(Pc, P_R=1e-12)
        #Save data
        R_had.append(Rf)
        M_had.append(Mf / Msun_in_km)

    R_had = np.array(R_had)
    M_had = np.array(M_had)

    curves.append(R_had)
    curves.append(M_had)
    headers += ["R_had", "M_had"]
        
    plt.plot(R_had, M_had, color='black',lw=2.2, zorder=10, label="Pure hadronic (Crust+APR-1)")

    #Hybrid curves (quark + hadronic)
    for (a4, D, Beff_14, Ptr) in hybrid_sets:

        Etr = float(APR_1(Ptr))
        DEcrit = 0.5*Etr + 1.5*Ptr
        dE_vals = [DEcrit + off for off in dE_values]

        for dE in dE_vals:
            #Hybrid EoS
            den_en_function = build_den_en_hybrid(a4, D, Beff_14, Ptr, dE, xm_min=900, xm_max=2500, N=8000,check_seidov=False)

            R_list, M_list = [], []
            for Pc in Pc_vals:
                try:
                    R_hyb, M_hyb = integrate_star_hybrid(Pc, den_en_function, P_R=1e-12)
                    R_list.append(R_hyb)
                    M_list.append(M_hyb / Msun_in_km)
                except ValueError:
                    break

            R_arr = np.array(R_list)
            M_arr = np.array(M_list)

            curves.append(R_arr)
            curves.append(M_arr)

            headers += [f"R_a4{a4}_D{D}_B{Beff_14}_Ptr{Ptr:.2f}_dE{dE:.1f}",f"M_a4{a4}_D{D}_B{Beff_14}_Ptr{Ptr:.2f}_dE{dE:.1f}"]

            plt.plot(R_list, M_list, label=f"Hybrid a4={a4},D={D},B14={Beff_14}, Ptr={Ptr:.2f}, dE={dE:.1f}")
         

    plt.xlim(6, 20)
    plt.grid(True)
    plt.legend(fontsize=8)
    plt.xlabel("Radius R (km)")
    plt.ylabel("Mass M / M$_\\odot$")
    plt.title("Hybrid Star: Quark (CFL) and Hadronic (APR-1) Matter")
    fig2.savefig('Hybrid-Hadronic Matter.png', dpi=80, bbox_inches='tight')
    plt.tight_layout()
    plt.show()

    max_len = max(len(arr) for arr in curves)

    padded = []
    for arr in curves:
        pad = np.full(max_len, np.nan)
        pad[:len(arr)] = arr
        padded.append(pad)

    data_out = np.column_stack(padded)

    np.savetxt(
        "MR_ALL_curves_multi_column.txt",
        data_out,
        fmt="%.8e",
        header="  ".join(headers),
        comments=""
    )

    print("Saved: MR_ALL_curves_multi_column.txt")


#Parameters (a4, D, Beff_14, Ptr)
hybrid_sets = [
(0.7,   0, 180, 209.54),
(0.7,  25, 170, 141.60),
(0.7,  50, 160, 5.80),
(0.7,  50, 170, 101.73),
(0.7,  50, 180, 171.88),
(0.7,  75, 170, 32.75),
(0.3, 150, 180, 200.95)] 

MR_curves(hybrid_sets)